In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [3]:
# 物流的渠道对照关系清洗用
month = 202508
Channel_map = {
'工程': '工程',
'零售':'零售',
'电商不可售':'电商',
'电商':'电商',
'内部处理通用':'电商',
'借出渠道':'无',
'新品':'电商',
'战略电商':'电商',
'转出渠道':'无',
'每誉':'每誉',
'渠道':'无',
'海外':'海外',
'调出渠道':'无',
'非零售工程电商':'非零售工程电商',
'无':'无'
}
#用于合并计算
productgroupset_map = {
    '吸油烟机':['吸油烟机'],
    '灶具':['灶具'],
    '烤箱':['烤箱'],
    '蒸箱':['蒸箱'],
    '微波炉':['微波炉'],
    '蒸烤烹饪机':['蒸烤烹饪机'],
    '蒸烤微烹饪机':['蒸烤微烹饪机'],
    '蒸微':['蒸微'],
    '蒸烤微合计':['烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微'],
    '灶消烹饪机':['灶消烹饪机'],
    '灶蒸烹饪机':['灶蒸烹饪机'],
    '灶蒸烤烹饪机':['灶蒸烤烹饪机'],
    '灶烤烹饪机':['灶烤烹饪机'],
    '灶集成':['灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机'],
    '烹饪产品线合计':['灶具','烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微','灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机'],
    '消毒柜':['消毒柜'],
    '热水器':['热水器'],
    '两用炉':['两用炉'],
    '热水器两用炉合计':['热水器','两用炉'],
    '家用净水机':['家用净水机'],
    '商用净水机':['商用净水机'],
    '净热产品线合计':['热水器','两用炉','家用净水机','商用净水机'],
    '水槽洗碗机':['水槽洗碗机'],
    '嵌入式洗碗机':['嵌入式洗碗机'],
    '洗碗机产品线合计':['水槽洗碗机','嵌入式洗碗机'],
    '国内合计':['吸油烟机','灶具','烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微','灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机','消毒柜','热水器','两用炉','家用净水机','商用净水机','水槽洗碗机','嵌入式洗碗机'],
}
# 统计的产品组
productgroup_list = ['吸油烟机','灶具','烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微','灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机','消毒柜','热水器','两用炉','热水器两用炉合计','家用净水机','商用净水机','净热产品线合计','水槽洗碗机','嵌入式洗碗机','洗碗机产品线合计','国内合计']


In [16]:
df = pd.read_excel(fr'D:\000物料报表\{month}\单物料产值\统计周期内产品核算价汇总.xlsx')
df_plm = pd.read_excel(fr"D:\000物料报表\202508\单型号贡献-低效-长尾\产品生命周期状态全表20250703.xlsx")


In [17]:
df.head()

,商品编码,渠道,实际出库数量,产品组,系统核算价,核算价,标准型号,国内/海外
0,1009001100002,工程,1,蒸烤烹饪机,3550.0,3550.0,ZK50-01-F1.i,国内
1,1001002100022,工程,2,吸油烟机,1508.0,3016.0,JC03A,国内
2,1002003400032,工程,2,灶具,900.0,1800.0,TH3B,国内
3,1001002000018,工程,1,吸油烟机,3668.0,3668.0,03-X1A,国内
4,1003000500029,工程,1,消毒柜,1550.0,1550.0,ZTD100J-J31,国内


In [18]:
df['商品编码'] = df['商品编码'].astype(str)
df_plm['物料号'] = df_plm['物料号'].astype(str)
chanpingxian = dict(zip(df_plm['物料号'], df_plm['产品线']))
chanpingxinghao = dict(zip(df_plm['物料号'], df_plm['产品型号']))
life = dict(zip(df_plm['物料号'], df_plm['产品状态']))


In [19]:
df['物料描述'] = df['商品编码'].map(chanpingxinghao)
df['产品线'] = df['商品编码'].map(chanpingxian)
df['生命周期'] = df['商品编码'].map(life)
df

,商品编码,渠道,实际出库数量,产品组,系统核算价,核算价,标准型号,国内/海外,物料描述,产品线,生命周期
0,1009001100002,工程,1,蒸烤烹饪机,3550.0,3550.0,ZK50-01-F1.i,国内,ZK50-01-F1.i,烹饪厨电产品线,退市预警
1,1001002100022,工程,2,吸油烟机,1508.0,3016.0,JC03A,国内,CXW-258-JC03A(不带罩),油烟机产品线,量产
2,1002003400032,工程,2,灶具,900.0,1800.0,TH3B,国内,JZT-TH33B-12T,烹饪厨电产品线,量产
3,1001002000018,工程,1,吸油烟机,3668.0,3668.0,03-X1A,国内,CXW-358-03-X1A,油烟机产品线,量产
4,1003000500029,工程,1,消毒柜,1550.0,1550.0,ZTD100J-J31,国内,ZTD100J-J51E,冰储产品线,量产
...,...,...,...,...,...,...,...,...,...,...,...
322374,1021000400003,非零售工程电商,240,地面手持式清洁机,4200.0,1008000.0,NaN,国内,QX-V7Pro-L,公共/战略,量产
322375,1021000400007,非零售工程电商,810,地面手持式清洁机,3522.0,2852820.0,NaN,国内,QX-V8-H,公共/战略,小批量
322376,1021000400008,非零售工程电商,150,地面手持式清洁机,3522.0,528300.0,QX-V8,国内,QX-V8-Z,公共/战略,量产
322377,1021000200003,非零售工程电商,300,地面手持式清洁机,3886.0,1165800.0,Q7Plus-L,国内,QX-Q7Plus-L,公共/战略,小批量


In [22]:
df['商品发货总数'] = df.groupby('商品编码')['实际出库数量'].transform('sum')
df1 = df[['商品编码','物料描述','产品组','产品线','生命周期','标准型号','商品发货总数','国内/海外']].drop_duplicates()
df1


,商品编码,物料描述,产品组,产品线,生命周期,标准型号,商品发货总数,国内/海外
0,1009001100002,ZK50-01-F1.i,蒸烤烹饪机,烹饪厨电产品线,退市预警,ZK50-01-F1.i,13781,国内
1,1001002100022,CXW-258-JC03A(不带罩),吸油烟机,油烟机产品线,量产,JC03A,14752,国内
2,1002003400032,JZT-TH33B-12T,灶具,烹饪厨电产品线,量产,TH3B,58539,国内
3,1001002000018,CXW-358-03-X1A,吸油烟机,油烟机产品线,量产,03-X1A,55760,国内
4,1003000500029,ZTD100J-J51E,消毒柜,冰储产品线,量产,ZTD100J-J31,13613,国内
...,...,...,...,...,...,...,...,...
321939,1005000700001,KQD60F-F1,烤箱,烹饪厨电产品线,停止生产,KQD60F-F1,3,国内
322032,1008000400008,JPSD2T-GD03,水槽洗碗机,洗碗机产品线,停止销售,JPSD2T-G3,2,国内
322178,1009000900001,JZT-ZK60-X3.i,灶蒸烤烹饪机,烹饪厨电产品线,停止发货,JZT-ZK60-X3.i,2,国内
322300,1013000100006,YCZ-JT120-M5A,家用净水机,净热产品线,停止发货,YCZ-JT120-M5A,1,国内


In [24]:
df1['生命周期'].value_counts()

生命周期
量产      949
停止销售    224
退市预警     90
小批量      69
开发       37
停止发货     33
样机       25
停止生产     18
作废        1
Name: count, dtype: int64

In [ ]:
life_vals_map = {
    '开发': 1,
    '样机': 2,
    '小批量': 3,
    '量产': 4,
    '退市预警': 5,
    '停止销售': 6,
    '停止生产': 7,
    '停止发货': 8,
    '作废': 9
}


# 1. 先给原表添加“生命周期状态的数字优先级”列
df1['生命周期_优先级'] = df1['生命周期'].map(life_vals_map)

# 2. 按“标准型号”分组，计算每个型号的最小优先级（最早状态）
early_stage_by_model = df1.groupby('标准型号')['生命周期_优先级'].min()

# 3. 将结果映射回原表（同一型号的所有行都会得到相同的“最早状态”）
df1['标准型号对应的最早状态'] = df1['标准型号'].map(early_stage_by_model)

# （可选）删除临时列
df1.drop('生命周期_优先级', axis=1, inplace=True)

In [29]:
reverse_life_map = {v: k for k, v in life_vals_map.items()}
# 将数字优先级转为文本状态
df1['标准型号对应的最早状态_文本'] = df1['标准型号对应的最早状态'].map(reverse_life_map)
df1

,商品编码,物料描述,产品组,产品线,生命周期,标准型号,商品发货总数,国内/海外,标准型号对应的最早状态,标准型号对应的最早状态_文本
0,1009001100002,ZK50-01-F1.i,蒸烤烹饪机,烹饪厨电产品线,退市预警,ZK50-01-F1.i,13781,国内,5.0,退市预警
1,1001002100022,CXW-258-JC03A(不带罩),吸油烟机,油烟机产品线,量产,JC03A,14752,国内,4.0,量产
2,1002003400032,JZT-TH33B-12T,灶具,烹饪厨电产品线,量产,TH3B,58539,国内,4.0,量产
3,1001002000018,CXW-358-03-X1A,吸油烟机,油烟机产品线,量产,03-X1A,55760,国内,4.0,量产
4,1003000500029,ZTD100J-J51E,消毒柜,冰储产品线,量产,ZTD100J-J31,13613,国内,4.0,量产
...,...,...,...,...,...,...,...,...,...,...
321939,1005000700001,KQD60F-F1,烤箱,烹饪厨电产品线,停止生产,KQD60F-F1,3,国内,7.0,停止生产
322032,1008000400008,JPSD2T-GD03,水槽洗碗机,洗碗机产品线,停止销售,JPSD2T-G3,2,国内,6.0,停止销售
322178,1009000900001,JZT-ZK60-X3.i,灶蒸烤烹饪机,烹饪厨电产品线,停止发货,JZT-ZK60-X3.i,2,国内,8.0,停止发货
322300,1013000100006,YCZ-JT120-M5A,家用净水机,净热产品线,停止发货,YCZ-JT120-M5A,1,国内,8.0,停止发货


In [30]:
df1.to_excel(r'C:\Users\zhangbon\Desktop\发货型号数据统计-0918.xlsx',index=False)
